### Deparse

In [1]:
import json
import pandas as pd
import re

input_file = "output20260414.json"
output_file = input_file + ".csv"

rows = []

# --- PROCESS CIBLES ---
targets = {
    "unbound": ["unbound"],
    "kworker": ["avahi-daemon"],
    "networkmanager": ["NetworkManager"],
    "ksoftirqd": ["ksoftirqd"]
}

with open(input_file, "r") as f:
    content = f.read()

# corrige JSON concaténé
objects = content.replace("}{", "}\n{").splitlines()

for obj in objects:
    try:
        j = json.loads(obj)

        # --- HOST ---
        timestamp = j["host"]["timestamp"]
        host_power = j["host"]["consumption"] / 1e6  # W

        # --- CPU (socket + domains) ---
        socket_power = 0.0
        core_power = 0.0
        uncore_power = 0.0

        for s in j.get("sockets", []):
            socket_power += s.get("consumption", 0.0)

            for d in s.get("domains", []):
                if d["name"] == "core":
                    core_power += d.get("consumption", 0.0)
                elif d["name"] == "uncore":
                    uncore_power += d.get("consumption", 0.0)

        socket_power /= 1e6
        core_power /= 1e6
        uncore_power /= 1e6

        # --- INIT METRICS ---
        metrics = {}
        for t in targets:
            metrics[t] = {
                "power": 0.0,
                "cpu": 0.0,
                "mem": 0.0,
                "disk_read": 0.0,
                "disk_write": 0.0
            }

        # --- AGGREGATION PROCESS ---
        for c in j.get("consumers", []):
            exe = c.get("exe", "")
            cmd = c.get("cmdline", "")

            for name, patterns in targets.items():
                if any(re.search(rf"\b{p}\b", exe) or re.search(rf"\b{p}\b", cmd) for p in patterns):
                    metrics[name]["power"] += c.get("consumption", 0.0)

                    r = c.get("resources_usage", {})

                    metrics[name]["cpu"] += float(r.get("cpu_usage", 0) or 0)
                    metrics[name]["mem"] += float(r.get("memory_usage", 0) or 0)
                    metrics[name]["disk_read"] += float(r.get("disk_usage_read", 0) or 0)
                    metrics[name]["disk_write"] += float(r.get("disk_usage_write", 0) or 0)

        # conversion en watts
        for t in metrics:
            metrics[t]["power"] /= 1e6

        # --- BUILD ROW ---
        row = {
            "timestamp": timestamp,
            "host_power_W": host_power,
            "cpu_socket_power_W": socket_power,
            "cpu_core_power_W": core_power,
            "cpu_uncore_power_W": uncore_power,
        }

        for t, m in metrics.items():
            row[f"{t}_power_W"] = m["power"]
            row[f"{t}_cpu_percent"] = m["cpu"]
            row[f"{t}_memory_bytes"] = m["mem"]
            row[f"{t}_disk_read_bytes"] = m["disk_read"]
            row[f"{t}_disk_write_bytes"] = m["disk_write"]

        rows.append(row)

    except Exception:
        # skip lignes corrompues
        pass

df = pd.DataFrame(rows)

if df.empty:
    raise ValueError("Aucune donnée valide parsée")

# conversion timestamp lisible
df["datetime"] = pd.to_datetime(df["timestamp"], unit="s")

df.to_csv(output_file, index=False)

print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'output20260414.json'

In [ ]:
# Mixage

import pandas as pd

def enrich_wattmeter(wattmeter_csv, scaphandre_csv, output_csv):
    # Charger les données
    watt = pd.read_csv(wattmeter_csv)
    scaph = pd.read_csv(scaphandre_csv)

    # S'assurer que les timestamps sont en float
    watt["time_begin"] = watt["time_begin"].astype(float)
    watt["time_end"] = watt["time_end"].astype(float)
    scaph["timestamp"] = scaph["timestamp"].astype(float)

    # Colonnes à agréger (toutes sauf timestamp + datetime)
    cols_to_avg = [col for col in scaph.columns if col not in ["timestamp", "datetime"]]

    # Liste pour stocker les résultats
    results = []

    # Boucle sur chaque ligne de wattmeter
    for _, row in watt.iterrows():
        t_start = row["time_begin"]
        t_end = row["time_end"]

        # Filtrage temporel
        subset = scaph[(scaph["timestamp"] >= t_start) & (scaph["timestamp"] <= t_end)]

        # Calcul des moyennes
        if not subset.empty:
            means = subset[cols_to_avg].mean()
        else:
            # Si aucune donnée dans l'intervalle
            means = pd.Series([float('nan')] * len(cols_to_avg), index=cols_to_avg)

        results.append(means)

    # Transformer en DataFrame
    means_df = pd.DataFrame(results)

    # Fusion avec wattmeter
    enriched = pd.concat([watt.reset_index(drop=True), means_df.reset_index(drop=True)], axis=1)

    # Sauvegarde
    enriched.to_csv(output_csv, index=False)

    return enriched


# Exemple d'utilisation
enrich_wattmeter("final_scaphandre_active_unbound_1c1t_local.csv", "output20260414.json.csv", "extend_final_scaphandre_active_unbound_1c1t_local.csv")

### Graph

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger le CSV
df = pd.read_csv("output20260408.json;csv")

# Convertir datetime si pas déjà fait
if "datetime" in df.columns:
    df["datetime"] = pd.to_datetime(df["datetime"])
    x = df["datetime"]
else:
    x = df["timestamp"]

# Plot
plt.figure()

plt.plot(x, df["host_power_W"], label="Host Power")
plt.plot(x, df["named_power_W"], label="Bind Power")
#plt.plot(x, df["cpu_socket_power_W"], label="CPU Socket Power")
plt.plot(x, df["cpu_core_power_W"], label="CPU Core Power")
plt.plot(x, df["cpu_uncore_power_W"], label="CPU Uncore Power")
#plt.plot(x, df["kworker_power_W"], label="KWorker Power")
#plt.plot(x, df["networkmanager_power_W"], label="NetworkManager Power")
#plt.plot(x, df["ksoftirqd_power_W"], label="ksoftirqd Power")

plt.xlabel("Time")
plt.ylabel("Power (W)")
plt.title("Power Consumption Over Time")

plt.legend()
plt.grid()

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'output20260408.json;csv'

## Traitement wattmètre

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mtick

def load_data(csv_file):
    """Charge les données du fichier csv et renvoie le DataFrame."""
    df = pd.read_csv(csv_file)
    return df

def list_slots(df):
    """Retourne la liste triée des numéros de slot présents dans le DataFrame."""
    return [int(x) for x in sorted(df["slot"].unique())]

def compute_avg_throughput(df):
    """
    Calcule pour chaque slot :
        - total_requests = somme des nb_request
        - total_time     = somme (time_end - time_begin)
        - avg_throughput       = total_requests / total_time

    Renvoie trois listes : slots, total_requests, avg_rates
    """

    df["duration"] = df["time_end"] - df["time_begin"]

    grouped = df.groupby("slot").agg(
        total_requests=("nb_request", "sum"),
        total_time=("duration", "sum")
    )
    #grouped's type = Dataframe

    grouped["avg_throughput"] = grouped["total_requests"] / grouped["total_time"]

    # Convertir en listes simples
    avg_throughput = grouped["avg_throughput"].tolist()

    return avg_throughput

def compute_avg_power(df):
    """
    Calcule pour chaque slot :
        - total_power = somme cumulative_power
        - total_data_wattmeter = somme nb_data_wattmeter_collected
        - avg_power       = total_power / total_data_wattmeter

    Renvoie trois listes : slots, total_requests, avg_rates
    """
    grouped = df.groupby("slot").agg(
        total_power=("cumulative_power", "sum"),
        total_data_wattmeter=("nb_data_wattmeter_collected", "sum")
    )
    #grouped's type = Dataframe

    grouped["avg_power"] = grouped["total_power"] / grouped["total_data_wattmeter"]

    # Convertir en listes simples
    avg_power = grouped["avg_power"].tolist()

    return avg_power

# Average delays in microseconds
def compute_avg_delay(df):
    """
    Calcule pour chaque slot :
        - total_delay_cumul = somme delay_resp_cumulative
        - total_resp     = somme nb_response
        - avg_delay       = total_delay_cumul / total_resp

    Renvoie trois listes : slots, total_requests, avg_rates
    """
    grouped = df.groupby("slot").agg(
        total_delay_cumul=("delay_resp_cumulative", "sum"),
        total_resp=("nb_response", "sum")
    )
    #grouped's type = Dataframe

    grouped["avg_delay"] = grouped["total_delay_cumul"] / grouped["total_resp"] #* 10**6

    # Convertir en listes simples
    avg_delay = grouped["avg_delay"].tolist()

    return avg_delay

# Average rates in %
def compute_avg_rates(df):
    """
    Calcule pour chaque slot :
        - total_requests = somme des nb_request
        - total_resp     = somme nb_response
        - avg_rates       = total_delay_cumul / total_resp

    Renvoie trois listes : slots, total_requests, avg_rates
    """
    grouped = df.groupby("slot").agg(
        total_requests=("nb_request", "sum"),
        total_resp=("nb_response", "sum")
    )
    #grouped's type = Dataframe

    grouped["avg_rates"] = grouped["total_resp"] / grouped["total_requests"] * 100

    # Convertir en listes simples
    avg_rates = grouped["avg_rates"].tolist()

    return avg_rates

def compute_std_power(df):

    grouped = df.groupby("slot").agg(
        sum_for_meanofvar   = ("var_power",  lambda x: (x * df.loc[x.index, "nb_data_wattmeter_collected"]).sum()), # sum(nb * varpw)
        sum_for_varofmeans  = ("mean_power", lambda x: (df.loc[x.index, "nb_data_wattmeter_collected"] * x**2).sum()), # sum(nb * meanpw**2)
        sum_for_meanofmeans = ("mean_power", lambda x: (df.loc[x.index, "nb_data_wattmeter_collected"] * x).sum()), # sum(nb * meanpw)
        n                   = ("nb_data_wattmeter_collected", "sum") # sum(nb)
    ).reset_index()

    grouped["meanofmeans"] = grouped["sum_for_meanofmeans"]/grouped["n"]     # ap = (j*a1+k*a2+m*a3)/n avec n=j+k+m - ap the pooled average
    grouped["meanofvar"] = grouped["sum_for_meanofvar"]/grouped["n"]     # a(v) = (j*v1+k*v2+m*v3)/n avec n=j+k+m
    grouped["varofmeans"] = grouped["sum_for_varofmeans"]/grouped["n"] - grouped["meanofmeans"]**2     # v(a) = (j*a1²+k*a2²+m*a3²)/n - ap²
    
    grouped["pooledvar"] = grouped["meanofvar"]+grouped["varofmeans"]     # vp=a(v)+v(a)
    pooled_std_power = np.sqrt(grouped["pooledvar"])

    return pooled_std_power, grouped["n"]

def compute_avg_host_power(df):
    df["duration"] = df["time_end"] - df["time_begin"]

    grouped = df.groupby("slot").agg(
        weighted_power=("host_power_W", lambda x: (x * df.loc[x.index, "duration"]).sum()),
        total_duration=("duration", "sum")
    )

    grouped["avg_host_power"] = grouped["weighted_power"] / grouped["total_duration"]

    return grouped["avg_host_power"].tolist()

def compute_avg_unbound_power(df):
    df["duration"] = df["time_end"] - df["time_begin"]

    grouped = df.groupby("slot").agg(
        weighted_power=("unbound_power_W", lambda x: (x * df.loc[x.index, "duration"]).sum()),
        total_duration=("duration", "sum")
    )

    grouped["avg_unbound_power"] = grouped["weighted_power"] / grouped["total_duration"]

    return grouped["avg_unbound_power"].tolist()

def compute_std_delay(df):

    grouped = df.groupby("slot").agg(
        sum_for_meanofvar   = ("var_delay",  lambda x: (x * df.loc[x.index, "nb_response"]).sum()), # sum(nb * varpw)
        sum_for_varofmeans  = ("mean_delay", lambda x: (df.loc[x.index, "nb_response"] * x**2).sum()), # sum(nb * meanpw**2)
        sum_for_meanofmeans = ("mean_delay", lambda x: (df.loc[x.index, "nb_response"] * x).sum()), # sum(nb * meanpw)
        n                   = ("nb_response", "sum") # sum(nb)
    ).reset_index()

    grouped["meanofmeans"] = grouped["sum_for_meanofmeans"]/grouped["n"]     # ap = (j*a1+k*a2+m*a3)/n avec n=j+k+m - ap the pooled average
    grouped["meanofvar"] = grouped["sum_for_meanofvar"]/grouped["n"]     # a(v) = (j*v1+k*v2+m*v3)/n avec n=j+k+m
    grouped["varofmeans"] = grouped["sum_for_varofmeans"]/grouped["n"] - grouped["meanofmeans"]**2     # v(a) = (j*a1²+k*a2²+m*a3²)/n - ap²
    
    grouped["pooledvar"] = grouped["meanofvar"]+grouped["varofmeans"]     # vp=a(v)+v(a)
    pooled_std_delay = np.sqrt(grouped["pooledvar"])

    return pooled_std_delay, grouped["n"]

def confidence_intervalle_99(std, n):
    
    # coef 2.58 pour 99% de confiance ; 1.96 pour 95% de confiance
    confidence_margin = 2.58 * std / np.sqrt(n)
    
    return confidence_margin

def plot_power(avg_throughput, avg_power, confidence_margin_power):
    
    plt.figure()

    #plt.plot(avg_throughput, avg_power, marker='o')
    
    plt.errorbar(
        avg_throughput,
        avg_power,
        yerr=confidence_margin_power,
        marker='o',
        capsize=5
    )
    
    plt.xlabel("Average throughput (qps)")
    plt.ylabel("Average power (W)")
    plt.title("Power as a function of throughput")
    plt.grid(True)
    plt.show()

def plot_delay(avg_throughput, avg_delay, confidence_margin_delay):
    plt.figure()

    #plt.plot(avg_throughput, avg_delay, marker='o')

    plt.errorbar(
        avg_throughput,
        avg_delay,
        yerr=confidence_margin_delay,
        marker='o',
        capsize=5
    )
    
    plt.xlabel("Average throughput (qps)")
    plt.ylabel("Average response time (s)")
    plt.title("Average response time as a function of throughput")
    plt.grid(True)
    plt.show()



def plot_rates(avg_throughput, avg_rates):
    fig, ax = plt.subplots()

    ax.plot(avg_throughput, avg_rates, marker='o')
    ax.set_xlabel("Average throughput (qps)")
    ax.set_ylabel("Average response rate (%)")
    ax.set_title("Response rate as a function of throughput")
    ax.grid(True)

    # Forcer l'affichage en pourcentage, sans notation scientifique, max 5 décimales
    ax.yaxis.set_major_formatter(
        mtick.FuncFormatter(lambda y, _: f"{y:.5f}%")
    )

    plt.show()

def main(datafile,P_idle, L, k, x0):
    
    df = load_data(datafile)

    # Traitement des données
    slots = list_slots(df)
    
    avg_throughput = compute_avg_throughput(df)
    avg_delay = compute_avg_delay(df)
    avg_rates = compute_avg_rates(df)
    avg_power = compute_avg_power(df)
    
    std_power, n_power = compute_std_power(df)
    std_delay, n_delay = compute_std_delay(df)
    
    confidence_margin_power = confidence_intervalle_99(std_power,n_power)
    confidence_margin_delay = confidence_intervalle_99(std_delay,n_delay)

    # Plots
    plot_power(avg_throughput, avg_power, confidence_margin_power)
    plot_delay(avg_throughput, avg_delay, confidence_margin_delay)
    plot_rates(avg_throughput, avg_rates)

    # Plot model
    plot_sigmoid_power(P_idle, L, k, x0, avg_throughput, avg_power, confidence_margin_power)

    

In [4]:
# Multiplot
from IPython.display import display

def compute_metrics(datafile):
    df = load_data(datafile)

    avg_throughput = compute_avg_throughput(df)
    avg_power = compute_avg_power(df)
    avg_host_power = compute_avg_host_power(df)
    avg_unbound_power = compute_avg_unbound_power(df)
    avg_delay = compute_avg_delay(df)

    std_power, n_power = compute_std_power(df)
    std_delay, n_delay = compute_std_delay(df)

    confidence_margin_power = confidence_intervalle_99(std_power, n_power)
    confidence_margin_delay = confidence_intervalle_99(std_delay, n_delay)

    return {
        "throughput": avg_throughput,
        "power": avg_power,
        "host_power": avg_host_power,
        "unbound_power" : avg_unbound_power,
        "delay": avg_delay,
        "ci_power": confidence_margin_power,
        "ci_delay": confidence_margin_delay,
    }




def display_slots_table(datafiles, labels=None):
   
    for i, datafile in enumerate(datafiles):
        metrics = compute_metrics(datafile)
        label = labels[i] if labels else datafile

        table = pd.DataFrame(
            {
                f"slot_{i+1}": [
                    metrics["throughput"][i],
                    metrics["power"][i],
                    metrics["delay"][i] * 1e3 ,
                ]
                for i in range(len(metrics["throughput"]))
            },
            index=["Throughput (qps)", "Power (W)", "Response time (ms)"]
        )
    
        display(table.style
                .format("{:.3f}")
                .set_caption(label)
                .set_table_styles([
                    {"selector": "caption", "props": [("font-size", "14px"),
                                                       ("font-weight", "bold")]}
                ]))

'''def multi_plot_power(datafiles, labels=None, colors=None, savename=None, formatfile=None):
    fig, ax = plt.subplots()

    for i, datafile in enumerate(datafiles):
        metrics = compute_metrics(datafile)
        label = labels[i] if labels else datafile

        ''''''ax.errorbar(
            metrics["throughput"],
            metrics["power"],
            yerr=metrics["ci_power"],
            marker='o',
            capsize=4,
            label=label
        )''''''

        kwargs = dict(
            yerr=metrics["ci_power"],
            marker='o',
            capsize=4,
            label=label
        )

        # ajoute la couleur seulement si fournie
        if colors is not None:
            kwargs["color"] = colors[i]

        ax.errorbar(
            metrics["throughput"],
            metrics["power"],
            **kwargs
        )

    ax.set_xlabel("Average throughput (qps)")
    ax.set_ylabel("Average power (W)")
    #ax.set_title("Power as a function of throughput")
    ax.grid(True)
    ax.legend()

    fig.canvas.draw()

    if savename is not None:
        if formatfile is None : 
            fig.savefig(f"images_exp/{savename}_power.pdf", bbox_inches="tight")
        else :
            fig.savefig(f"images_exp/{savename}_power.{formatfile}", bbox_inches="tight")

    plt.show()
    plt.close(fig)
'''

def multi_plot_power(datafiles, labels=None, colors=None, savename=None, formatfile=None):
    fig, ax = plt.subplots()

    for i, datafile in enumerate(datafiles):
        metrics = compute_metrics(datafile)
        label = labels[i] if labels else datafile

        color = colors[i] if colors is not None else None

        # --- puissance wattmètre (avec IC) ---
        ax.errorbar(
            metrics["throughput"],
            metrics["power"],
            yerr=metrics["ci_power"],
            marker='o',
            capsize=4,
            label=f"{label} (wattmeter)",
            color=color
        )

        # --- puissance scaphandre (sans IC) ---
        '''ax.plot(
            metrics["throughput"],
            metrics["host_power"],
            linestyle='--',
            marker='x',
            label=f"{label} (scaphandre)",
            color=color
        )'''

        ax.plot(
            metrics["throughput"],
            np.array(metrics["host_power"])*10,
            linestyle='--',
            marker='x',
            label=f"{label} (host scaphandre réhaussé)",
            color=color
        )

        #unbound_power
        ax.plot(
            metrics["throughput"],
            np.array(metrics["unbound_power"])*10,
            linestyle='--',
            marker='x',
            label=f"{label} (unbound scaphandre réhaussé)",
            color=color
        )

    ax.set_xlabel("Average throughput (qps)")
    ax.set_ylabel("Average power (W)")
    ax.grid(True)

    #ax.legend()
    # légende à droite
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    
    # laisse de la place
    fig.subplots_adjust(right=0.75)

    fig.canvas.draw()

    if savename is not None:
        if formatfile is None:
            fig.savefig(f"images_exp/{savename}_power.pdf", bbox_inches="tight")
        else:
            fig.savefig(f"images_exp/{savename}_power.{formatfile}", bbox_inches="tight")

    plt.show()
    plt.close(fig)

def multi_plot_delay(datafiles, labels=None, colors=None, savename=None, formatfile=None):
    fig, ax = plt.subplots()

    for i, datafile in enumerate(datafiles):
        metrics = compute_metrics(datafile)
        label = labels[i] if labels else datafile

        '''ax.errorbar(
            metrics["throughput"],
            np.array(metrics["delay"]) * 1e3,
            yerr=np.array(metrics["ci_delay"]) * 1e3,
            marker='o',
            capsize=4,
            label=label
        )'''
        
        kwargs = dict(
            yerr=np.array(metrics["ci_delay"]) * 1e3,
            marker='o',
            capsize=4,
            label=label
        )

        # ajoute la couleur seulement si fournie
        if colors is not None:
            kwargs["color"] = colors[i]

        ax.errorbar(
            metrics["throughput"],
            np.array(metrics["delay"]) * 1e3,
            **kwargs
        )

    ax.set_xlabel("Average throughput (qps)")
    ax.set_ylabel("Average response time (ms)")
    #ax.set_title("Average response time as a function of throughput")
    ax.grid(True)
    ax.legend()

    fig.canvas.draw()
    
    if savename is not None:
        if formatfile is None : 
            fig.savefig(f"images_exp/{savename}_delay.pdf", bbox_inches="tight")
        else :
            fig.savefig(f"images_exp/{savename}_delay.{formatfile}", bbox_inches="tight")
            
    plt.show()
    plt.close(fig)


def multi_plot(datafiles, labels=None, colors=None, savename=None, formatfile=None):
    display_slots_table(datafiles, labels)

    multi_plot_power(datafiles, labels, colors, savename, formatfile)
    multi_plot_delay(datafiles, labels, colors, savename, formatfile)


In [5]:
datafiles = [
    "aextend_final_scaphandre_active_unbound_1c1t_local.csv",
    "aextend_final_scaphandre_active_unbound_4c2t_local.csv",
]

labels = [
    "1c1t",
    "4c2t",
]

'''colors = [
    "#A9CCE3", #bleu pâle
    "#1F77B4", #bleu saturé
    "#F8C471", #orange pâle
    "#E67E22", #orange saturé
    "#A9DFBF", #vert pâle
    #"#28B463", #vert saturé
    "#F5B7B1", #rouge pâle
    #"#C0392B", #rouge saturé
]'''

multi_plot(datafiles, labels)#, None, "resolvers_local_1c1t", "svg")


FileNotFoundError: [Errno 2] No such file or directory: 'aextend_final_scaphandre_active_unbound_1c1t_local.csv'